# Study 809 — Signed Jump Variation ⚡📉

**Do stocks whose recent variance is *downside*-dominated go on to earn *more*?**

Barndorff-Nielsen, Kinnebrock & Shephard (2010) split realized variance by the **sign of the
return** — upside `RS+ = Σ r²·1(r>0)` vs downside `RS- = Σ r²·1(r<0)` — and Bollerslev, Li &
Zhao (2020) show the **signed jump variation** `SJ = (RS+ − RS-)/RV` is priced *negatively*:
the **downside** ("bad" volatility) names carry a **premium**, the **upside** ("good"
volatility) names under-earn. A long **low-SJ** / short **high-SJ** book should earn a positive
spread. We take the self-contained daily version on a liquid US cross-section
(2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

Split a month's daily variance into the part from **up** days (`RS+`, 'good' volatility) and the part from **down** days (`RS-`, 'bad' volatility). The signed jump `SJ = (RS+ − RS-)/RV` is positive when the wobble is mostly on the upside, negative when it is mostly on the downside. The theory: downside volatility is genuinely feared, so bearing it is *paid* — buy the downside-heavy names, sell the lottery-like upside-heavy ones.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-1.71, t_nw=-1.36, lo_bps=7.2, hi_bps=8.91, gross_sharpe=-0.32)
print('long low-SJ / short high-SJ spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  low-SJ (downside) book %+.2f bps vs high-SJ (upside) book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long low-SJ / short high-SJ spread: -1.71 bps/day (NW t = -1.36)
  low-SJ (downside) book +7.20 bps vs high-SJ (upside) book +8.91 bps
  gross spread Sharpe (before cost): -0.32


## 2. Is the sort just noise? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, signed jump present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from signed_jump import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=809, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0024, seed=809, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = -1.43  (should be ~0)
planted world: spread NW t = +3.66  (should light up)


## 3. The honest verdict — the famous edge does *not* replicate here

On this liquid mega-cap tape the long-low-SJ / short-high-SJ spread is **-1.71 bps/day** with NW *t* = **-1.36** — **not significant** (|*t*| < 2), and the point estimate even leans the **wrong way**: the upside-dominated ('good' volatility) names, if anything, *out-earned* the downside names (the permutation null centres at 0 with sd 0.90 bps; the observed value is only ~1.9σ into the *left* tail). The seeded synthetic control recovers a *planted* Bollerslev-Li-Zhao relation cleanly, so this is a genuine **absence** on the mega-cap survivor universe, not a bug — the signed-jump premium is a smaller-cap phenomenon that does not survive on 50 mega-caps. **Signal: None** (the claimed edge is absent), **Tradability: Mirage**.